# 📄 Cheeni Capability 6: Document & Receipt OCR
### Microsoft TrOCR (`microsoft/trocr-base-printed`)

**Core Capability**: Optical Character Recognition for scanned receipts, documents, invoices, and system dialogs.
**Foundation Architecture**: Transformer OCR (ViT spatial patch encoder + RoBERTa autoregressive text generation decoder).
**Benchmark Corpus**: Consolidated Receipt Dataset (`naver-clova-ix/cord-v2`).
**Target Hugging Face Hub**: `wraithklu/cheeni-ocr-trocr_06`
**Compute Environment**: Google Colab (Tesla T4 GPU or CPU)

In [ ]:
# 1. Install Vision OCR & Hugging Face Dependencies
!pip install -q transformers datasets evaluate accelerate Pillow huggingface_hub

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
login()

In [ ]:
# 3. Load Microsoft TrOCR Architecture
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_ID = "microsoft/trocr-base-printed"
print(f"⚡ Loading TrOCR architecture ({MODEL_ID})...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print(f"✅ TrOCR model loaded on {device}!")

In [ ]:
# 4. Stream CORD-v2 Real Document Benchmark
from datasets import load_dataset

print("📥 Streaming CORD-v2 Receipt benchmark dataset...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="validation")
print(f"✅ Total validation document images: {len(dataset)}")

# Display sample receipt details
sample_image = dataset[0]["image"].convert("RGB")
print(f"Sample Document Resolution: {sample_image.size}")

In [ ]:
# 5. Test Live OCR Text Extraction on Benchmark Document
# TrOCR extracts lines and text regions with ViT patches
# Crop top region of receipt (e.g. store name / item header)
w, h = sample_image.size
header_crop = sample_image.crop((0, int(h * 0.15), w, int(h * 0.35)))

pixel_values = processor(images=header_crop, return_tensors="pt").pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(pixel_values, max_new_tokens=64)

extracted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("=" * 60)
print(f"🎯 Recognized Document Text: '{extracted_text}'")
print("=" * 60)

In [ ]:
# 6. Push Verified TrOCR Model & Processor to Hugging Face Hub
HF_REPO_ID = "wraithklu/cheeni-ocr-trocr_06"

print(f"Pushing trained OCR model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID)
processor.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")